#### 문항 1. 무신사 상품 10페이지

무신사 상품 페이지에서 상의 카테고리 제품을 10페이지 크롤링해주세요.

* 대상: https://www.musinsa.com/category/001/goods?gf=A

* 추출 필드: 브랜드명 / 제품명 / 원래가격 / 할인가격 / 리뷰 수 / 리뷰 점수

* Selenium 혹은 requests로 수집

* Selenium으로 수집 시

    * time.sleep()으로 요소를 기다리지 말 것. WebDriverWait + expected_conditions를 사용할 것

    * headless 모드로 실행하고 창 크기를 명시할 것

    * try / finally로 드라이버가 반드시 종료되게 할 것

* 결과를 musinsa.csv로 저장할 것

기대 결과
```
[{'브랜드명': '생로랑', '제품명': '클래식 반소매 티셔츠 - 블랙 / 801474YB2FT1000',
  '원래가격': 838000, '할인가격': 105990, '리뷰수': 1, '리뷰점수': 100},
 {'브랜드명': '108파운드', '제품명': 'Days Comfort Fit Shirt_White',
  '원래가격': 89000, '할인가격': 76100, '리뷰수': 0, '리뷰점수': 0}, ...]
```

In [ ]:
import csv
import re
from collections import Counter

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException

BASE_URL = "https://www.musinsa.com/category/001/goods?gf=A"
TOTAL_PAGES = 10
OUTPUT_FILE = "musinsa.csv"
WAIT_SECONDS = 15


def build_driver():
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=options)


def to_int(price_text: str) -> int:
    """'53,100원' -> 53100 처럼 숫자만 남긴다."""
    digits = re.sub(r"[^\d]", "", price_text or "")
    return int(digits) if digits else 0


def find_item_container(link_element, max_depth: int = 6):
    node = link_element
    for _ in range(max_depth):
        try:
            node = node.find_element(By.XPATH, "..")
        except NoSuchElementException:
            break
        if "원" in node.text:
            return node
    return node


def extract_original_price(driver, container):
    for el in container.find_elements(By.XPATH, ".//*[contains(text(), '원')]"):
        try:
            deco = driver.execute_script(
                "return window.getComputedStyle(arguments[0]).textDecorationLine;", el
            )
        except Exception:
            continue
        if deco and "line-through" in deco:
            return to_int(el.text)
    return None


def extract_sale_price(container, original_price):
    candidates = []
    for el in container.find_elements(By.XPATH, ".//*[contains(text(), '원')]"):
        price = to_int(el.text)
        if price and price != original_price:
            candidates.append(price)
    if candidates:
        return candidates[-1]
    return original_price or 0


def extract_review(container):
    match = re.search(r"(\d+\.\d+)\s*\((\d+)\)", container.text)
    if not match:
        return 0, 0
    star = float(match.group(1))
    count = int(match.group(2))
    score = round(star * 20)
    return count, score


def extract_brand_and_name(container):
    links = container.find_elements(By.TAG_NAME, "a")
    texts = [a.text.strip() for a in links if a.text.strip()]
    if not texts:
        return "", ""

    counts = Counter(texts)
    repeated = [t for t, c in counts.items() if c >= 2]
    if repeated:
        name = repeated[0]
        others = [t for t in dict.fromkeys(texts) if t != name]
        brand = others[0] if others else ""
    else:
        uniq = list(dict.fromkeys(texts))
        brand = uniq[0] if uniq else ""
        name = uniq[1] if len(uniq) >= 2 else brand
    return brand, name


def scrape_page(driver, page_no: int):
    url = f"{BASE_URL}&page={page_no}"
    driver.get(url)

    wait = WebDriverWait(driver, WAIT_SECONDS)
    wait.until(
        EC.presence_of_all_elements_located(
            (By.XPATH, "//a[contains(@href, '/products/')]")
        )
    )

    product_links = driver.find_elements(By.XPATH, "//a[contains(@href, '/products/')]")

    seen_ids = set()
    rows = []
    for link in product_links:
        href = link.get_attribute("href") or ""
        m = re.search(r"/products/(\d+)", href)
        if not m:
            continue
        goods_no = m.group(1)
        if goods_no in seen_ids:
            continue
        seen_ids.add(goods_no)

        container = find_item_container(link)
        brand, name = extract_brand_and_name(container)
        original_price = extract_original_price(driver, container)
        sale_price = extract_sale_price(container, original_price)
        review_count, review_score = extract_review(container)

        rows.append(
            {
                "브랜드명": brand,
                "제품명": name,
                "원래가격": original_price if original_price else sale_price,
                "할인가격": sale_price,
                "리뷰수": review_count,
                "리뷰점수": review_score,
            }
        )
    return rows


def main():
    driver = build_driver()
    all_rows = []
    try:
        for page in range(1, TOTAL_PAGES + 1):
            print(f"{page} 페이지 수집 중...")
            rows = scrape_page(driver, page)
            print(f"  -> {len(rows)}개 상품 수집")
            all_rows.extend(rows)
    finally:
        driver.quit()

    fieldnames = ["브랜드명", "제품명", "원래가격", "할인가격", "리뷰수", "리뷰점수"]
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"총 {len(all_rows)}개 상품을 {OUTPUT_FILE} 에 저장했습니다.")


if __name__ == "__main__":
    main()

1 페이지 수집 중...
  -> 23개 상품 수집
2 페이지 수집 중...
  -> 23개 상품 수집
3 페이지 수집 중...
  -> 23개 상품 수집
4 페이지 수집 중...
  -> 23개 상품 수집
5 페이지 수집 중...
  -> 23개 상품 수집
6 페이지 수집 중...
  -> 23개 상품 수집
7 페이지 수집 중...
  -> 23개 상품 수집
8 페이지 수집 중...
  -> 23개 상품 수집
9 페이지 수집 중...
  -> 23개 상품 수집
10 페이지 수집 중...
  -> 23개 상품 수집
총 230개 상품을 musinsa.csv 에 저장했습니다.


#### 문항 2. 로켓펀치 채용공고 10페이지

로켓펀치 채용 페이지에서 채용공고를 10페이지 수집하시오.

* 대상: https://www.rocketpunch.com/jobs

* 추출 필드: 기업명 / 공고명 / 요약 / 업무형태

* 결과를 rocketpunch.csv로 저장할 것

결과 예시
```
[{'기업명': '딥세일즈',
  '공고명': 'B2B 세일즈 인턴',
  '요약': '리드 데이터 및 캠페인 운영',
  '업무형태': '출근-재택 혼합'},
 {'기업명': '아이엠랩',
  '공고명': '[4년 이상] 필드 엔지니어 세일즈',
  '요약': '필드 세일즈 및 유통망 전략 전문가 모집',
  '업무형태': '상시 출근'}, ...]
```

In [ ]:
import csv
import time
import uuid

import requests

API_URL = "https://www.rocketpunch.com/api/proxy/jobs"
TOTAL_PAGES = 10
OUTPUT_FILE = "rocketpunch.csv"

BASE_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json",
    "x-rocket-api-version": "3.0.0",
    "x-rocket-app-key": "077cd9d8-7237-4bee-b2d3-77690d163cca",
    "x-rocket-client-type": "WEB",
    "x-rocket-device-type": "PC",
    "x-rocket-os-type": "WIN",
    "Referer": "https://www.rocketpunch.com/jobs",
    "Origin": "https://www.rocketpunch.com",
}


def build_headers():
    headers = dict(BASE_HEADERS)
    headers["x-rocket-request-id"] = str(uuid.uuid4())
    return headers


def fetch_page(page_token=None):
    params = {"sort": "DATE_DESC"}
    if page_token:
        params["pageToken"] = page_token

    response = requests.get(API_URL, headers=build_headers(), params=params, timeout=10)
    response.raise_for_status()
    return response.json()


def extract_rows(data):
    rows = []
    for job in data.get("items", []):
        rows.append(
            {
                "기업명": job.get("companyName", ""),
                "공고명": job.get("title", ""),
                "요약": job.get("description", ""),
                "업무형태": job.get("workType", ""),
            }
        )
    return rows


def main():
    all_rows = []
    page_token = None

    for page in range(1, TOTAL_PAGES + 1):
        print(f"{page} 페이지 요청 중...")
        data = fetch_page(page_token)

        rows = extract_rows(data)
        print(f"  -> {len(rows)}개 공고 수집 (전체 {data.get('totalItems', '?')}건 중)")
        all_rows.extend(rows)

        page_token = data.get("pageToken")
        if not page_token:
            print("다음 페이지 토큰이 없습니다. 여기서 멈춥니다.")
            break

        time.sleep(0.5) 

    fieldnames = ["기업명", "공고명", "요약", "업무형태"]
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"총 {len(all_rows)}개 공고를 {OUTPUT_FILE} 에 저장했습니다.")


if __name__ == "__main__":
    main()

1 페이지 요청 중...
  -> 20개 공고 수집 (전체 491건 중)
2 페이지 요청 중...
  -> 20개 공고 수집 (전체 491건 중)
3 페이지 요청 중...
  -> 20개 공고 수집 (전체 491건 중)
4 페이지 요청 중...
  -> 20개 공고 수집 (전체 491건 중)
5 페이지 요청 중...
  -> 20개 공고 수집 (전체 491건 중)
6 페이지 요청 중...
  -> 20개 공고 수집 (전체 491건 중)
7 페이지 요청 중...
  -> 20개 공고 수집 (전체 491건 중)
8 페이지 요청 중...
  -> 20개 공고 수집 (전체 491건 중)
9 페이지 요청 중...
  -> 20개 공고 수집 (전체 491건 중)
10 페이지 요청 중...
  -> 20개 공고 수집 (전체 491건 중)
총 200개 공고를 rocketpunch.csv 에 저장했습니다.
